## `░ 1. Instalación de librerías`

In [ ]:
# %pip install sqlalchemy
# %pip install pandas
# %pip install oracledb 
%pip install openpyxl

#### `» 1.1 Importar librerías`

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
import oracledb

## `░ 2. Configuración y métodos globales`

#### `» 2.1 Credenciales y parámetros de conexión`

In [3]:
# Conexión:

USER = '<USUARIO>'
PASSWORD = '<PASSWORD>'
HOST = '<HOST>'
PORT = '1521'
SID = 'INABIF02'

#### `» 2.2 Conexión a la base de datos Oracle`

In [4]:
def get_engine():
   dsn = oracledb.makedsn(HOST, PORT, sid=SID)
   connection_string = f"oracle+oracledb://{USER}:{PASSWORD}@{dsn}"
   return create_engine(
      connection_string,
      poolclass=NullPool,  # Sin pool para evitar problemas
      echo=False)


#### `» 2.3 Métodos globales del workflow`

In [5]:
# 2.3.1 Cache de metadata de Oracle y helper de schema
_SCHEMA_CACHE = {}

def obtener_metadata_tabla(table_name, engine=None):
   key = table_name.upper()
   if key in _SCHEMA_CACHE:
      return _SCHEMA_CACHE[key]
   if engine is None:
      engine = get_engine()
   q = text("""
      SELECT COLUMN_NAME, DATA_TYPE, DATA_SCALE, NULLABLE
      FROM USER_TAB_COLUMNS
      WHERE TABLE_NAME = :t
   """)
   with engine.connect() as c:
      rows = c.execute(q, {'t': key}).fetchall()
   meta = {
      'date':        {r[0] for r in rows if r[1] == 'DATE'},
      'numeric_int': {r[0] for r in rows if r[1] == 'NUMBER' and (r[2] is None or r[2] == 0)},
      'numeric_dec': {r[0] for r in rows if r[1] == 'NUMBER' and r[2] is not None and r[2] > 0},
      'not_null':    {r[0] for r in rows if r[3] == 'N'},
      'columns':     {r[0] for r in rows},
   }
   _SCHEMA_CACHE[key] = meta
   return meta

# 2.3.2 Parsear serie a datetime (formato fijo DD/MM/YYYY)
def parsear_fecha_serie(serie):
   if pd.api.types.is_datetime64_any_dtype(serie):
      return serie
   if serie.dtype != object:
      return serie
   return pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce')

# 2.3.3 Normalizar DataFrame para Oracle (schema-driven, generico)
def normalizar_dataframe_para_oracle(df, table_name, engine):
   meta = obtener_metadata_tabla(table_name, engine)
   df_normalizado = df.copy()

   for columna in df_normalizado.columns:
      serie = df_normalizado[columna]
      if columna in meta['date']:
         parseada = parsear_fecha_serie(serie)
         df_normalizado[columna] = parseada.where(parseada.notna(), None)
      elif columna in meta['numeric_int']:
         numerica = pd.to_numeric(serie, errors='coerce').astype('Int64')
         df_normalizado[columna] = numerica.where(numerica.notna(), None)
      elif columna in meta['numeric_dec']:
         numerica = pd.to_numeric(serie, errors='coerce')
         df_normalizado[columna] = numerica.where(numerica.notna(), None)
      elif serie.dtype == object:
         s_limpia = serie.where(serie.notna(), '').astype(str).str.strip()
         df_normalizado[columna] = serie.where(s_limpia != '', None)

   return df_normalizado

# 2.3.4 Filtrar filas incompletas antes de persistir (schema-driven, generico)
def filtrar_filas_invalidas_para_oracle(df, table_name, engine, strict=False):
   meta = obtener_metadata_tabla(table_name, engine)
   df_filtrado = df.copy()

   mascara_validos = pd.Series(True, index=df_filtrado.index)
   for columna in meta['not_null']:
      if columna in df_filtrado.columns:
         serie = df_filtrado[columna]
         if pd.api.types.is_datetime64_any_dtype(serie):
            vacia = serie.isna()
         else:
            s_limpia = serie.where(serie.notna(), '').astype(str).str.strip()
            vacia = serie.isna() | (s_limpia == '')
         mascara_validos &= ~vacia

   invalidos = df_filtrado[~mascara_validos]
   if not invalidos.empty:
      cols_log = [c for c in meta['not_null'] if c in df_filtrado.columns]
      print(f'[WARN] {table_name}: {len(invalidos)} filas descartadas por NOT NULL vacios')
      print(invalidos[cols_log].head(5).to_string(index=True))
      if strict:
         raise ValueError(f'{len(invalidos)} filas invalidas en {table_name} (strict=True)')

   df_filtrado = df_filtrado[mascara_validos].dropna(how='all')
   return df_filtrado

# 2.3.5 Helpers de validacion y preparacion del workflow
def validar_identificador_oracle(nombre, tipo='identificador'):
   valor = str(nombre).strip().upper()
   valor_simple = valor.replace('_', '')
   if not valor or not valor[0].isalpha() or not valor_simple.isalnum():
      raise ValueError(f'{tipo} Oracle invalido: {nombre}')
   return valor

def normalizar_columnas_dataframe(df):
   df_normalizado = df.copy()
   df_normalizado.columns = [str(c).strip().upper() for c in df_normalizado.columns]
   duplicadas = df_normalizado.columns[df_normalizado.columns.duplicated()].tolist()
   if duplicadas:
      raise ValueError(f'Columnas duplicadas despues de normalizar: {duplicadas}')
   return df_normalizado

def validar_columnas_dataframe_para_tabla(df, table_name, engine, id_column=None):
   meta = obtener_metadata_tabla(table_name, engine)
   if not meta['columns']:
      raise ValueError(f'No se encontro metadata para la tabla {table_name}')

   columnas_df = set(df.columns)
   columnas_desconocidas = sorted(columnas_df - meta['columns'])
   if columnas_desconocidas:
      raise ValueError(f'{table_name}: columnas no existen en Oracle: {columnas_desconocidas}')

   if id_column is not None and id_column not in columnas_df:
      raise ValueError(f'{table_name}: la columna ID {id_column} no existe en el Excel/DataFrame')

   return True

def convertir_valor_para_bind(valor):
   if pd.isna(valor):
      return None
   if isinstance(valor, pd.Timestamp):
      return valor.to_pydatetime()
   if hasattr(valor, 'item'):
      return valor.item()
   return valor

def dataframe_a_registros_bind(df):
   registros = []
   for fila in df.to_dict(orient='records'):
      registros.append({k: convertir_valor_para_bind(v) for k, v in fila.items()})
   return registros

# 2.3.6 Insertar DataFrame por lotes a la base de datos Oracle
def insertar_dataframe_por_lotes(df, table_name, lote_size=1000, strict=False):
   engine = None
   try:
      engine = get_engine()
      df_filtrado = filtrar_filas_invalidas_para_oracle(df, table_name, engine, strict=strict)
      df_normalizado = normalizar_dataframe_para_oracle(df_filtrado, table_name, engine)
      with engine.begin() as conn:
         df_normalizado.to_sql(name=table_name, con=conn, if_exists='append', index=False, chunksize=lote_size)
      invalidos = len(df) - len(df_normalizado)
      print(f'[OK] {table_name}: insertados={len(df_normalizado)} de_origen={len(df)} invalidos={invalidos}')
      return True, len(df_normalizado)
   except Exception as e:
      print(f'[ERROR] {table_name}: {e}')
      return False, 0
   finally:
      if engine is not None:
         engine.dispose()

# 2.3.7 Actualizar DataFrame por lotes usando una columna ID
def actualizar_dataframe_por_lotes(df, table_name, id_column, lote_size=1000, strict=False, dry_run=False):
   engine = None
   try:
      table_name = validar_identificador_oracle(table_name, 'tabla')
      id_column = validar_identificador_oracle(id_column, 'columna ID')
      engine = get_engine()

      df_preparado = normalizar_columnas_dataframe(df)
      validar_columnas_dataframe_para_tabla(df_preparado, table_name, engine, id_column=id_column)

      mascara_id_valido = df_preparado[id_column].notna() & (df_preparado[id_column].astype(str).str.strip() != '')
      df_sin_id = df_preparado[~mascara_id_valido]
      if not df_sin_id.empty:
         print(f'[WARN] {table_name}: {len(df_sin_id)} filas omitidas por ID vacio en {id_column}')
         if strict:
            raise ValueError(f'{len(df_sin_id)} filas sin ID en {table_name} (strict=True)')

      df_filtrado = df_preparado[mascara_id_valido].dropna(how='all')
      df_normalizado = normalizar_dataframe_para_oracle(df_filtrado, table_name, engine)
      columnas_update = [c for c in df_normalizado.columns if c != id_column]
      if not columnas_update:
         raise ValueError(f'{table_name}: no hay columnas para actualizar aparte de {id_column}')

      asignaciones = ', '.join([f'{c} = :{c}' for c in columnas_update])
      sentencia = text(f'UPDATE {table_name} SET {asignaciones} WHERE {id_column} = :{id_column}')
      registros = dataframe_a_registros_bind(df_normalizado[[id_column] + columnas_update])

      if dry_run:
         print(f'[DRY_RUN] {table_name}: operacion=update id={id_column} filas_validas={len(registros)} columnas_update={len(columnas_update)}')
         print(f'[DRY_RUN] columnas_update={columnas_update}')
         return True, {'procesadas': len(registros), 'actualizadas': 0, 'omitidas_sin_id': len(df_sin_id), 'dry_run': True}

      actualizadas = 0
      with engine.begin() as conn:
         for inicio in range(0, len(registros), lote_size):
            lote = registros[inicio:inicio + lote_size]
            resultado = conn.execute(sentencia, lote)
            if resultado.rowcount and resultado.rowcount > 0:
               actualizadas += resultado.rowcount

      no_encontradas = max(len(registros) - actualizadas, 0)
      print(f'[OK] {table_name}: actualizadas={actualizadas} procesadas={len(registros)} omitidas_sin_id={len(df_sin_id)} no_encontradas_estimadas={no_encontradas}')
      return True, {'procesadas': len(registros), 'actualizadas': actualizadas, 'omitidas_sin_id': len(df_sin_id), 'no_encontradas_estimadas': no_encontradas}
   except Exception as e:
      print(f'[ERROR] {table_name}: {e}')
      return False, {'procesadas': 0, 'actualizadas': 0, 'error': str(e)}
   finally:
      if engine is not None:
         engine.dispose()

# 2.3.8 Wrapper para insertar o actualizar desde el mismo workflow
def procesar_dataframe_por_lotes(df, table_name, operacion='insert', id_column=None, lote_size=1000, strict=False, dry_run=False):
   operacion = str(operacion).strip().lower()
   if operacion == 'insert':
      if dry_run:
         engine = get_engine()
         try:
            df_preparado = normalizar_columnas_dataframe(df)
            validar_columnas_dataframe_para_tabla(df_preparado, validar_identificador_oracle(table_name, 'tabla'), engine)
            df_filtrado = filtrar_filas_invalidas_para_oracle(df_preparado, table_name, engine, strict=strict)
            print(f'[DRY_RUN] {table_name}: operacion=insert filas_validas={len(df_filtrado)} de_origen={len(df)}')
            return True, len(df_filtrado)
         finally:
            engine.dispose()
      return insertar_dataframe_por_lotes(df, table_name, lote_size=lote_size, strict=strict)

   if operacion == 'update':
      if not id_column:
         raise ValueError('Para operacion=update debes indicar id_column')
      return actualizar_dataframe_por_lotes(df, table_name, id_column=id_column, lote_size=lote_size, strict=strict, dry_run=dry_run)

   raise ValueError(f'Operacion no soportada: {operacion}. Usa insert o update')

# 2.3.9 Ejecutar consulta y retornar DataFrame
def execute_query(query, params=None):
   engine = None
   try:
      engine = get_engine()
      with engine.connect() as conn:
         if params:
            df = pd.read_sql_query(sql=text(query), con=conn, params=params)
         else:
            df = pd.read_sql_query(sql=text(query), con=conn)
      print(f'Consulta ejecutada exitosamente, filas encontradas: {len(df)}')
      return df
   except Exception as e:
      print(f"Error al ejecutar la consulta: {e}")
      return pd.DataFrame()
   finally:
      if engine is not None:
         engine.dispose()

## `░ 3. Workflow de carga masiva desde Excel`

#### `» 3.1 Rutas de archivos Excel`

In [10]:
FILE_PATH_COMPOSICION = './source/punche/4_composicion_familiar_resuelto_bulk.xlsx'
FILE_PATH_FICHA_IDENTIFICACION = './source/punche/3_ficha_identificacion_unpivot.xlsx'
FILE_PATH_FICHA_COMPROMISO = './source/punche/5_compromiso_familiar_unpivot.xlsx'
FILE_PATH_FICHA_FFSIL = './source/punche/6_ficha_ffsil_unpivot.xlsx'
FILE_PATH_FICHA_DIAGNOSTICO = './source/punche/7_ficha_diagnostico_unpivot.xlsx'
FILE_PATH_FICHA_TSV = './source/punche/8_ficha_tsv_unpivot.xlsx'
FILE_PATH_CATALOGOS = 'C:/Users/uti.rguevara_01/Desktop/data_fichas_inabif v1/ssi_catalogos.xlsx'
FILE_PATH_UPDATED_FECREG_FAMILIA = r'C:\Users\uti.rguevara_01\Downloads\cod_familia_+_id_Familia.xlsx'

#### `» 3.2 Workflow parametrizado insert/update`

In [11]:
WORKFLOW_CONFIG = {
   'file_path': FILE_PATH_UPDATED_FECREG_FAMILIA,
   'sheet_name': 'PF_FEC_REGISTRA',
   'table_name': 'SSI_POTENCIALES_FAMILIAS',
   'operacion': 'insert',  # insert | update
   'id_column': None,      # requerido si operacion=update
   'lote_size': 1000,
   'strict': False,
   'dry_run': True,        # cambiar a False para ejecutar cambios
}

def cargar_excel_desde_config(config):
   print(f"[INFO] Cargando Excel: {config['file_path']} | hoja={config['sheet_name']}")
   return pd.read_excel(config['file_path'], sheet_name=config['sheet_name'])

def ejecutar_workflow_desde_config(config):
   df_origen = cargar_excel_desde_config(config)
   return procesar_dataframe_por_lotes(
      df_origen,
      config['table_name'],
      operacion=config['operacion'],
      id_column=config['id_column'],
      lote_size=config['lote_size'],
      strict=config['strict'],
      dry_run=config['dry_run'])

# Ejemplo update:
# WORKFLOW_CONFIG.update({'operacion': 'update', 'id_column': 'AP_ID_PREGUNTA', 'dry_run': True})
# ejecutar_workflow_desde_config(WORKFLOW_CONFIG)

#### `» 3.3 Ejecución principal del workflow`

In [ ]:
# Ejecutar primero con dry_run=True para validar columnas, tipos y filas procesables.
# Cambiar WORKFLOW_CONFIG['dry_run'] = False solo cuando la validacion sea correcta.
# resultado_workflow = ejecutar_workflow_desde_config(WORKFLOW_CONFIG)
# resultado_workflow

#### `» 3.4 Ejemplos de configuración insert/update`

In [13]:
# 3.4.1 Insercion por lotes:
# WORKFLOW_CONFIG.update({
#    'operacion': 'insert',
#    'id_column': None,
#    'dry_run': True,
# })
# ejecutar_workflow_desde_config(WORKFLOW_CONFIG)

# 3.4.2 Actualizacion por ID:
WORKFLOW_CONFIG.update({
   'operacion': 'update',
   'id_column': 'PF_ID_FAMILIA',
   'dry_run': False,
})
ejecutar_workflow_desde_config(WORKFLOW_CONFIG)

[INFO] Cargando Excel: C:\Users\uti.rguevara_01\Downloads\cod_familia_+_id_Familia.xlsx | hoja=PF_FEC_REGISTRA
[OK] SSI_POTENCIALES_FAMILIAS: actualizadas=3264 procesadas=3264 omitidas_sin_id=0 no_encontradas_estimadas=0


(True,
 {'procesadas': 3264,
  'actualizadas': 3264,
  'omitidas_sin_id': 0,
  'no_encontradas_estimadas': 0})

#### `» 3.5 Ejemplos históricos de carga manual`

In [26]:
# Referencia historica: cargas manuales desde Excel.
# respuestas_composicion = pd.read_excel(FILE_PATH_COMPOSICION, sheet_name='_')
# respuestas_ficha_identificacion = pd.read_excel(FILE_PATH_FICHA_IDENTIFICACION, sheet_name='SIGEIFUnpivot')
# respuestas_ficha_compromiso = pd.read_excel(FILE_PATH_FICHA_COMPROMISO, sheet_name='SIGEIFUnpivot')
# respuestas_ficha_ffsil = pd.read_excel(FILE_PATH_FICHA_FFSIL, sheet_name='SIGEIFUnpivot')
# respuestas_ficha_diagnostico = pd.read_excel(FILE_PATH_FICHA_DIAGNOSTICO, sheet_name='SIGEIFUnpivot')
# respuestas_ficha_tsv = pd.read_excel(FILE_PATH_FICHA_TSV, sheet_name='SIGEIFUnpivot')
# preguntas_catalogo = pd.read_excel(FILE_PATH_CATALOGOS, sheet_name='EDUCALLE')

In [ ]:
# insertar_dataframe_por_lotes(respuestas_ficha_identificacion, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)
# insertar_dataframe_por_lotes(respuestas_ficha_compromiso, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)
# insertar_dataframe_por_lotes(respuestas_composicion, 'SSI_FAMILIA_INTEGRANTES', lote_size=1000)
# insertar_dataframe_por_lotes(respuestas_ficha_diagnostico, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)
# insertar_dataframe_por_lotes(respuestas_ficha_tsv, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)
# insertar_dataframe_por_lotes(preguntas_catalogo, 'SSI_ANEXOS_PREGUNTAS', lote_size=1000)

#### `» 3.5.1 Ejemplo histórico: Ficha 10 FF-SIL`

In [ ]:
# Referencia historica: importar datos desde Excel.
# respuestas_anexo_10 = pd.read_excel(FILE_PATH_MATRIZ, sheet_name='2.2 FUNC FAM PIVOT')

In [ ]:
# insertar_dataframe_por_lotes(respuestas_anexo_10, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

#### `» 3.5.2 Ejemplo histórico: Ficha 11 TSV`

In [ ]:
# Referencia historica: importar datos desde Excel.
# respuestas_anexo_11 = pd.read_excel(FILE_PATH_MATRIZ, sheet_name='3.2 TSV_PIVOT')

In [ ]:
# insertar_dataframe_por_lotes(respuestas_anexo_11, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

#### `» 3.5.3 Ejemplo histórico: Ficha 27`

In [ ]:
# Referencia historica: importar datos desde Excel.
# respuestas_anexo_27 = pd.read_excel(FILE_PATH_FICHA_27, sheet_name='1.1 Inabif_Act_3_Encuest_PIVOT')

In [ ]:
# insertar_dataframe_por_lotes(respuestas_anexo_27, 'SSI_ANEXOS_RESPUESTAS', lote_size=1000)

## `░ 4. Consultas de verificación`

In [ ]:
# 4.1 Consulta puntual de verificacion
QUERY = "SELECT * FROM SSI_ANEXOS_RESPUESTAS r WHERE r.AR_ID_RESPUESTA = :anexo_id"
df_result = execute_query(QUERY, params={'anexo_id': 1050})
df_result

In [ ]:
test_result = execute_query('''

               SELECT pv.* FROM (

                     SELECT
                        (
                           SELECT cf2.CF_CODIGO FROM (

                              SELECT 
                                 cf.CF_CODIGO,
                                 ROW_NUMBER() OVER (ORDER BY cf.CF_ID_CODIGO DESC) AS ORDEN_COD
                              FROM SSI_CODIGOS_FAMILIAS cf 
                              WHERE 
                                 cf.PF_ID_FAMILIA = f.PF_ID_FAMILIA
                                 AND cf.CF_ESTADO = 1
                                 AND cf.CF_ELIMINADO = 0

                           ) cf2
                           WHERE cf2.ORDEN_COD = 1

                        ) AS COD_FAM,

                        '-' AS NRO_VIV,

                        (
                           SELECT COUNT(1) FROM SSI_FAMILIA_INTEGRANTES i
                           WHERE 
                              i.FI_ESTADO = 1
                              AND i.FI_ELIMINADO = 0
                              AND i.FI_CUIDADOR != 1 -- * Todos excepto el cuidador
                              AND i.PF_ID_FAMILIA = f.PF_ID_FAMILIA
                        ) AS NUM_MIEM_FAM,

                        tf.CATDESCRIPCION AS TIP_FAM,
                        i.FI_TELEFONO AS NRO_TLF,	
                        td.CATDESCRIPCION AS TIP_DOC,	
                        i.FI_NUMERO_DOC AS NRO_DOC,	
                        i.FI_PRIMER_APE AS PRI_APE_FAM,	
                        i.FI_SEGUNDO_APE AS SEG_APE_FAM,	
                        i.FI_NOMBRES AS NOM_FAM,	
                        ts.CATDESCRIPCION AS SEXO,	
                        i.FI_FEC_NAC AS FEC_NAC,	
                        i.FI_EDAD AS EDAD_USU,	
                        ec.CATDESCRIPCION AS EST_CIV,
                        tp.CATDESCRIPCION AS PAR_FAM,	
                        '-' AS PAI_FAM,	

                        (
                           CASE
                              WHEN CA_TIENE_DISCAPACIDAD = 1 THEN 'SI'
                              ELSE 'NO'
                           END
                        ) AS TIE_DIS_FAM,	

                        '-' AS REG_CONADIS,
                        cd.CATDESCRIPCION AS TIP_DISC,
                        lm.CATDESCRIPCION AS LEN_MAT_FAM,	
                        '-' AS LEN_MAT_ESP_FAM,	
                        '-' AS AUT_IDE_ET_FAM,	
                        '-' AS AUT_IDE_ET_ESP_FAM,	
                        gi.CATDESCRIPCION AS GRAD_INST,	
                        co .CATDESCRIPCION AS OCU_FAM,	
                        ss .CATDESCRIPCION AS TIP_SEG_SAL,

                        -- * Fichas: 9, 10, 11
                        'Ficha-' || LPAD(p.AP_NUM_ANEXO, 2, '0') || ': ' || TRIM(p.AP_PREGUNTA) AS PREGUNTA,
                        r.AR_RESPUESTA AS RESPUESTA,
                        TO_CHAR(f.PF_FEC_REGISTRA, 'DD/MM/YYYY') AS FEC_REGISTRO_FICHA

                     FROM SSI_ZONA_INTERVENCION z
                     JOIN SSI_POTENCIALES_FAMILIAS f ON z.ZO_ID_ZONA = f.ZO_ID_ZONA
                     JOIN SSI_FAMILIA_INTEGRANTES i ON f.PF_ID_FAMILIA = i.PF_ID_FAMILIA
                     LEFT JOIN TGCATALOGO tf ON i.CA_ID_TIPO_FAMILIA = tf.IDCATALOGO
                     LEFT JOIN TGCATALOGO td ON i.CA_ID_TIPDOC = td.IDCATALOGO
                     LEFT JOIN TGCATALOGO ts ON i.CA_ID_SEXO = ts.IDCATALOGO
                     LEFT JOIN TGCATALOGO ec ON i.CA_ID_ESTADO_CIVIL = ec.IDCATALOGO
                     LEFT JOIN TGCATALOGO tp ON i.CA_ID_PARENTESCO = tp.IDCATALOGO
                     LEFT JOIN TGCATALOGO cd ON i.CA_ID_DISCAPACIDAD = cd.IDCATALOGO
                     LEFT JOIN TGCATALOGO lm ON i.CA_ID_LENGUA_MATERNA = lm.IDCATALOGO
                     LEFT JOIN TGCATALOGO gi ON i.CA_ID_GRADO_INST = gi.IDCATALOGO
                     LEFT JOIN TGCATALOGO co ON i.CA_ID_OCUPACION = co.IDCATALOGO
                     LEFT JOIN TGCATALOGO ss ON i.CA_ID_TIPO_SEGURO = ss.IDCATALOGO
                     JOIN SSI_ANEXOS_RESPUESTAS r ON f.PF_ID_FAMILIA = r.PF_ID_FAMILIA
                     JOIN SSI_ANEXOS_PREGUNTAS p ON p.AP_ID_PREGUNTA = r.AP_ID_PREGUNTA
                     JOIN SSI_ANEXO_FASES af ON p.SI_ID_SERVICIO = af.SF_ID_SERVICIO
                                             AND p.AP_NUM_ANEXO = af.SF_NUM_ANEXO
                                             AND r.SF_ID_FASE = af.SF_ID_FASE
                     WHERE 
                        f.PF_ESTADO = 1
                        AND f.PF_ELIMINADO = 0
                        AND i.FI_ESTADO = 1
                        AND i.FI_ELIMINADO = 0
                        AND i.FI_CUIDADOR = 1 -- * Cuidador principal
                        AND p.SI_ID_SERVICIO = 2 -- * PUNCHE
                        AND p.AP_NUM_ANEXO IN (9, 10, 11) -- * Diagnostico familiar, FFSIL y TSV
                        AND p.AP_NUM_GRUPO = -4 -- * Grupo resultados

                  ) f
                  PIVOT (
                     MAX(RESPUESTA)
                     FOR PREGUNTA IN ('Ficha-09: DIAGNOSTICO','Ficha-10: CALIFICACIÓN','Ficha-11: CALIFICACIÓN')
                  ) pv
                  ORDER BY pv.FEC_REGISTRO_FICHA ASC                            
                            
''')

In [ ]:
test_result.to_excel('./data/test_result_output.xlsx', index=False)